# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A — The Content Performance Curve (Finding #2) / ML Feature Importance appendix

The finding: Health score follows a lifecycle curve by content age, peaking at 61-90 days and dropping sharply by 271-365 days.

My methodology question — where does the label come from?

The paper's own ML appendix flags this concern for a related model: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." Health Score is explicitly built from Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — and the correlation matrix shows Average Position correlates at -0.6 with Health Score, by far the strongest relationship in the table. That raises a fair question: how much of the age→health curve reflects genuine content lifecycle decay, versus simply restating "older pages tend to have worse position" through a composite metric that already has position baked into it 30%? The paper is transparent about this limitation in the appendix, but the same caution arguably deserves a one-line callout directly on Finding #2's page, since it's presented as a headline result rather than exploratory.

Finding B — What Predicts Growth (Logistic Regression, 71% holdout accuracy)

The finding: Content age is the strongest negative predictor of growth; days visible and recent impressions are the strongest positive predictors.

My methodology question — does the validation design support the claim?

The methodology section states this model used an 80/20 split, but doesn't specify what the split was grouped or stratified by. Given the dataset spans 57 brands with potentially many pages per brand, a plain random 80/20 row split could place pages from the same brand in both train and test sets — letting the model partially learn brand-specific patterns (a strong brand's general "house style" of aging/updating) rather than a genuinely portfolio-wide growth signal. This is the same grouped-split concern I addressed in my own Week 5 model. A brand-grouped (or time-aware) holdout would make the 71% accuracy figure considerably more convincing as a generalizable pattern rather than partially reflecting known brands.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from dotenv import load_dotenv
import os
import duckdb

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

feature_df = con.sql(f"""
    WITH base AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_avg_position,
            gsc_impressions,
            gsc_clicks,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE month = '2026-03'
          AND gsc_avg_position IS NOT NULL
          AND gsc_impressions > 0
    ),
    bucket_avg AS (
        SELECT AVG(ctr) AS overall_avg_ctr FROM base
    )
    SELECT
        base.*,
        CASE WHEN base.ctr < bucket_avg.overall_avg_ctr THEN 1 ELSE 0 END AS y_low_ctr
    FROM base, bucket_avg
""").df()

feature_df.shape

(3611061, 8)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.